# 🦀 Day 4: Request Handling — Path, Query, JSON Bodies

---

## Extractors

Axum uses **extractors** to pull data from requests into handler parameters. Add them as function arguments — Axum injects them automatically.

| Extractor | Purpose |
|-----------|----------|
| `Path<T>` | Path parameters (e.g. `/users/:id`) |
| `Query<T>` | Query string (e.g. `?page=1&limit=10`) |
| `Json<T>` | JSON request body |
| `HeaderMap` | All request headers |
| `Request` | Raw request |

In [ ]:
:dep axum = "0.7"
:dep tokio = { version = "1", features = ["full"] }
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"

// Dependencies loaded.

## Path Parameters with Path<T>

Use `:name` in the route and `Path<T>` in the handler. The struct fields must match the path segment names.

In [ ]:
use axum::{Router, extract::Path, routing::get};

// Path params: /users/42 or /items/rust-book
async fn get_user(Path(id): Path<u32>) -> String {
    format!("User id: {}", id)
}

// Multiple path params: use a struct
#[derive(serde::Deserialize)]
struct ItemPath {
    category: String,
    id: String,
}

async fn get_item(Path(params): Path<ItemPath>) -> String {
    format!("Category: {}, Id: {}", params.category, params.id)
}

let app = Router::new()
    .route("/users/:id", get(get_user))
    .route("/items/:category/:id", get(get_item));

println!("Path extractors: Path<u32> for single, Path<Struct> for multiple.");

## Query Parameters with Query<T>

Parse `?key=value&foo=bar` into a struct with `Query<T>`. All fields are optional by default unless you use `serde(default)` or require them.

In [ ]:
use axum::{Router, extract::Query, routing::get};
use serde::Deserialize;

#[derive(Debug, Deserialize)]
struct Pagination {
    page: Option<u32>,
    limit: Option<u32>,
}

async fn list_items(Query(pagination): Query<Pagination>) -> String {
    let page = pagination.page.unwrap_or(1);
    let limit = pagination.limit.unwrap_or(10);
    format!("Page {}, limit {}", page, limit)
}

let app = Router::new().route("/items", get(list_items));

// Simulate: GET /items?page=2&limit=20
println!("Query extractor: Query<Pagination> parses ?page=2&limit=20");

## JSON Request Bodies with Json<T>

Use `Json<T>` to deserialize the request body. `T` must implement `DeserializeOwned` (e.g. via `#[derive(Deserialize)]`).

In [ ]:
use axum::{Router, extract::Json, routing::post};
use serde::{Deserialize, Serialize};

#[derive(Debug, Deserialize)]
struct CreateItem {
    name: String,
    price: f64,
}

#[derive(Serialize)]
struct ItemResponse {
    id: u32,
    name: String,
    price: f64,
}

async fn create_item(Json(body): Json<CreateItem>) -> Json<ItemResponse> {
    Json(ItemResponse {
        id: 1,
        name: body.name,
        price: body.price,
    })
}

let app = Router::new().route("/items", post(create_item));

println!("Json<CreateItem> extracts and parses the JSON body.");

## Combining Extractors

You can use multiple extractors in one handler. Order doesn't matter — Axum injects each by type.

In [ ]:
use axum::{Router, extract::{Path, Query, Json}, routing::{get, post}};
use serde::{Deserialize, Serialize};

#[derive(Deserialize)]
struct UpdateUser {
    name: Option<String>,
    email: Option<String>,
}

#[derive(Deserialize)]
struct QParams {
    verbose: Option<bool>,
}

async fn update_user(
    Path(id): Path<u32>,
    Query(q): Query<QParams>,
    Json(body): Json<UpdateUser>,
) -> String {
    format!("Update user {}: {:?}, verbose={:?}", id, body, q.verbose)
}

let app = Router::new().route("/users/:id", post(update_user));

println!("Path + Query + Json in one handler.");

## Response Types: Json<T>, (StatusCode, Json<T>)

Return `Json<T>` for 200 OK with JSON. Return `(StatusCode, Json<T>)` for errors or other status codes.

In [ ]:
use axum::{Router, extract::Path, Json, http::StatusCode, routing::get};
use serde::Serialize;

#[derive(Serialize)]
struct ErrorBody {
    error: String,
    code: u16,
}

async fn get_or_error(Path(id): Path<u32>) -> Result<Json<serde_json::Value>, (StatusCode, Json<ErrorBody>)> {
    if id == 0 {
        Err((
            StatusCode::BAD_REQUEST,
            Json(ErrorBody {
                error: "Invalid id".to_string(),
                code: 400,
            }),
        ))
    } else {
        Ok(Json(serde_json::json!({ "id": id })))
    }
}

// Result<T, E> where E: IntoResponse works as a response
let _app = Router::new().route("/items/:id", get(get_or_error));
println!("Result<Json<T>, (StatusCode, Json<E>)> for conditional responses.");

## Building CRUD Endpoint Handlers

Typical REST pattern:

| Method | Route | Handler |
|--------|-------|----------|
| GET | /items | list_items |
| GET | /items/:id | get_item |
| POST | /items | create_item |
| PUT | /items/:id | update_item |
| DELETE | /items/:id | delete_item |

In [ ]:
use axum::{Router, extract::{Path, Json}, routing::{get, post, put, delete}, http::StatusCode};
use serde::{Deserialize, Serialize};

#[derive(Debug, Clone, Serialize, Deserialize)]
struct Item {
    id: u32,
    name: String,
    price: f64,
}

#[derive(Deserialize)]
struct CreateItem {
    name: String,
    price: f64,
}

async fn list_items() -> Json<Vec<Item>> {
    Json(vec![])
}

async fn get_item(Path(id): Path<u32>) -> Result<Json<Item>, StatusCode> {
    if id == 0 { return Err(StatusCode::NOT_FOUND); }
    Ok(Json(Item { id, name: "Sample".into(), price: 9.99 }))
}

async fn create_item(Json(body): Json<CreateItem>) -> (StatusCode, Json<Item>) {
    let item = Item { id: 1, name: body.name, price: body.price };
    (StatusCode::CREATED, Json(item))
}

async fn update_item(Path(id): Path<u32>, Json(body): Json<CreateItem>) -> Json<Item> {
    Json(Item { id, name: body.name, price: body.price })
}

async fn delete_item(Path(id): Path<u32>) -> StatusCode {
    let _ = id;
    StatusCode::NO_CONTENT
}

let app = Router::new()
    .route("/items", get(list_items).post(create_item))
    .route("/items/:id", get(get_item).put(update_item).delete(delete_item));

println!("CRUD router for /items and /items/:id");

## 🏋️ Exercise

Write handlers for a **users** resource: `GET /users`, `GET /users/:id`, `POST /users`, `PUT /users/:id`. Use in-memory storage (a `Vec` or `HashMap`) for simplicity. Return appropriate status codes (201 for create, 404 for not found).

In [ ]:
// YOUR CODE HERE
//
// struct User { id: u32, name: String, email: String }
// Create Router with users routes
// Use std::sync::Arc<std::sync::Mutex<Vec<User>>> for shared state (we'll cover this in the next lesson)
// For now, you can use a simple Vec in each handler (no persistence between requests)
//
// Hint: Path<u32>, Query<...>, Json<...> as extractors

## 🎯 Key Takeaways

1. **Path<T>** for path params (`/users/:id`)
2. **Query<T>** for query string (`?page=1&limit=10`)
3. **Json<T>** for JSON request bodies
4. Combine extractors in one handler — order doesn't matter
5. Response types: `Json<T>`, `(StatusCode, Json<T>)`, `Result<T, E>` where E: IntoResponse
6. CRUD: GET list, GET one, POST create, PUT update, DELETE

---

**Tomorrow:** Middleware and shared state! 🔧